In [7]:
import pandas as pd
import numpy as np
from scipy import stats

df = pd.read_csv('dataset.csv')
print(df.shape)
df.head(5)

(55, 36)


,seed,mse_3_text_audio_prompt1,mse_7_text_audio_prompt1,mse_15_text_audio_prompt1,mse_30_text_audio_prompt1,avg_mse_text_audio_prompt1,mse_3_volonly,mse_7_volonly,mse_15_volonly,mse_30_volonly,...,mse_3_chunk_text_audio,mse_7_chunk_text_audio,mse_15_chunk_text_audio,mse_30_chunk_text_audio,avg_mse_chunk_text_audio,mse_3_random_nonqa,mse_7_random_nonqa,mse_15_random_nonqa,mse_30_random_nonqa,avg_mse_random_nonqa
0,1,0.318579,0.179663,0.191532,0.124573,0.203587,0.327742,0.194389,0.217952,0.149001,...,0.318962,0.180661,0.192716,0.126622,0.204740,0.318493,0.179451,0.191141,0.124290,0.203344
1,3,0.328327,0.186912,0.209081,0.142029,0.216587,0.341752,0.208540,0.247903,0.175780,...,0.330252,0.190228,0.215876,0.147225,0.220895,0.329063,0.188098,0.211407,0.143910,0.218119
2,5,0.319672,0.182200,0.186646,0.123136,0.202914,0.332046,0.202187,0.222952,0.149891,...,0.320754,0.184904,0.190331,0.125624,0.205403,0.319760,0.182296,0.186938,0.123360,0.203089
3,6,0.326771,0.179254,0.206812,0.157769,0.217652,0.338656,0.189462,0.232836,0.188170,...,0.327381,0.180008,0.207770,0.160535,0.218924,0.327988,0.180509,0.210624,0.161523,0.220161
4,9,0.331075,0.175289,0.199189,0.131390,0.209236,0.352457,0.193665,0.243874,0.165693,...,0.332965,0.176588,0.204781,0.133681,0.212004,0.331501,0.175718,0.200273,0.132187,0.209920


In [8]:
def summarize(df, winner, others):
    """Print mean MSE (sorted) and winner-vs-other paired-t results."""
    cols = [winner] + others
    print(f"n = {len(df)} seeds")
    print()
    print("Mean, sorted (lower = better):")
    print(df[cols].mean().sort_values())
    print()
    print(f"{'model':30s} {'mean diff':>12s} {'p-value':>10s}")
    for m in others:
        d = (df[m] - df[winner]).values
        t, p_two = stats.ttest_1samp(d, 0)
        p_one = p_two / 2 if t > 0 else 1 - p_two / 2
        print(f"{m:30s} {d.mean():12.5f} {p_one:10.4f}")

In [ ]:
h = 30
others = [f"mse_{h}_text_audio_prompt1"] + [f"mse_{h}_{target}" for target in ["audio_only", "text_only", "chunk_text_audio", "volonly"]]
summarize(df, f"mse_{h}_text_audio_prompt2", others)

# horizons: 3, 7, 15, 30

n = 55 seeds

Mean, sorted (lower = better):
mse_15_text_audio_prompt2    0.196477
mse_15_text_audio_prompt1    0.196601
mse_15_audio_only            0.197175
mse_15_text_only             0.197701
mse_15_chunk_text_audio      0.201971
mse_15_volonly               0.227504
dtype: float64

model                             mean diff    p-value
mse_15_text_audio_prompt1           0.00012     0.0525
mse_15_audio_only                   0.00070     0.0293
mse_15_text_only                    0.00122     0.0522
mse_15_chunk_text_audio             0.00549     0.0000
mse_15_volonly                      0.03103     0.0000


### Pairwise comparison of all models

All-pairs paired one-sided t-test between models.

In [16]:
all_models = [
    f"mse_{h}_text_audio_prompt1",
    f"mse_{h}_text_audio_prompt2",
    f"mse_{h}_audio_only",
    f"mse_{h}_text_only",
    f"mse_{h}_chunk_text_audio",
    f"mse_{h}_volonly",
]

short_names = {m: m.replace(f"mse_{h}_", "") for m in all_models}

mean_diff = pd.DataFrame(index=all_models, columns=all_models, dtype=float)
p_value = pd.DataFrame(index=all_models, columns=all_models, dtype=float)

for row in all_models:
    for col in all_models:
        if row == col:
            mean_diff.loc[row, col] = 0.0
            p_value.loc[row, col] = np.nan
            continue
        d = (df[col] - df[row]).values  # positive => row beats col
        t, p_two = stats.ttest_1samp(d, 0)
        p_one = p_two / 2 if t > 0 else 1 - p_two / 2
        mean_diff.loc[row, col] = d.mean()
        p_value.loc[row, col] = p_one

mean_diff = mean_diff.rename(index=short_names, columns=short_names)
p_value = p_value.rename(index=short_names, columns=short_names)

print(f"n = {len(df)} seeds")
print()
print("Mean diff (row - beats -> col; positive = row is better):")
print(mean_diff.round(5))
print()
print("One-sided p-value (row beats col):")
print(p_value.round(4))

n = 55 seeds

Mean diff (row - beats -> col; positive = row is better):
                    text_audio_prompt1  text_audio_prompt2  audio_only  \
text_audio_prompt1             0.00000            -0.00000     0.00072   
text_audio_prompt2             0.00000             0.00000     0.00072   
audio_only                    -0.00072            -0.00072     0.00000   
text_only                     -0.00003            -0.00003     0.00069   
chunk_text_audio              -0.00255            -0.00255    -0.00183   
volonly                       -0.01461            -0.01461    -0.01389   

                    text_only  chunk_text_audio  volonly  
text_audio_prompt1    0.00003           0.00255  0.01461  
text_audio_prompt2    0.00003           0.00255  0.01461  
audio_only           -0.00069           0.00183  0.01389  
text_only             0.00000           0.00251  0.01457  
chunk_text_audio     -0.00251           0.00000  0.01206  
volonly              -0.01457          -0.01206  0.0000